# Results and Paper Figures

Generate publication-quality figures for the paper:
1. Qualitative results grid
2. Confusion matrix
3. Baseline comparison
4. Cross-crater comparison
5. Full-strip segmentation overlay

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
import json
import torch
from pathlib import Path
import sys

sys.path.insert(0, str(Path('..').resolve()))
from src.train import build_model, build_dataloaders
from src.evaluate import Evaluator, compute_all_metrics
from src.visualize import (
    plot_qualitative_results, plot_confusion_matrix,
    plot_baseline_comparison, plot_cross_crater_comparison
)

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load Trained Model

In [ ]:
model = build_model(config, device)
checkpoint_path = Path('../outputs/checkpoints/best_model.pth')

if checkpoint_path.exists():
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Model loaded successfully")
else:
    print("No trained model found. Run training first.")

## 2. Qualitative Results

In [ ]:
_, val_loader = build_dataloaders(config)

model.eval()
all_patches, all_targets, all_preds = [], [], []

with torch.no_grad():
    for patches, masks in val_loader:
        patches = patches.to(device)
        probs = torch.sigmoid(model(patches))
        
        all_patches.append(patches.cpu().numpy().squeeze(1))
        all_targets.append(masks.numpy().squeeze(1))
        all_preds.append(probs.cpu().numpy().squeeze(1))

all_patches = np.concatenate(all_patches)
all_targets = np.concatenate(all_targets)
all_preds = np.concatenate(all_preds)

print(f"Total validation patches: {len(all_patches)}")

In [ ]:
plot_qualitative_results(
    all_patches, all_targets, all_preds,
    n_samples=6,
    output_path='../outputs/figures/qualitative_results.png'
)

## 3. Confusion Matrix

In [ ]:
pred_binary = (all_preds > 0.5).astype(np.uint8).flatten()
target_binary = (all_targets > 0.5).astype(np.uint8).flatten()

plot_confusion_matrix(
    pred_binary, target_binary,
    output_path='../outputs/figures/confusion_matrix.png'
)

## 4. Evaluation Metrics

In [ ]:
evaluator = Evaluator()
evaluator.update(all_preds, all_targets)
evaluator.print_summary('Final Model Evaluation')

results = evaluator.compute_aggregate()
with open('../outputs/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved to outputs/results.json")

## 5. Summary Table

In [ ]:
print("\n" + "="*70)
print("PAPER-QUALITY RESULTS")
print("="*70)
print(f"{'Metric':<20} {'Mean':>10} {'Std':>10} {'Target':>10} {'Status':>10}")
print("-"*70)

targets = {'iou': 0.70, 'dice': 0.80, 'pixel_accuracy': 0.90, 'hd95': 5.0, 'boundary_f1': 0.50}

for metric, stats in results.items():
    if metric in targets:
        mean = stats['mean']
        std = stats['std']
        target = targets[metric]
        if metric == 'hd95':
            status = 'PASS' if mean < target else 'FAIL'
        else:
            status = 'PASS' if mean > target else 'FAIL'
        print(f"{metric:<20} {mean:>10.4f} {std:>10.4f} {target:>10.4f} {status:>10}")

print("="*70)